In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# SC-SSTW Core Integration V1

Run all cells once. This fixed candidate creates four fresh Wan prompt/seed cases, freezes calibration from two independent OFF sources, then evaluates OFF, SINGLE46, and MULTI44_46 on two new sources carrying payloads `0x5` and `0xa`. It writes 56 saved views and performs 224 four-phase receiver encodes with the bound Partial3-V2 receiver. No mode or strength scan is exposed.

In [ ]:
import importlib.metadata, subprocess, sys
print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
subprocess.run([sys.executable, '-m', 'pip', '--version'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
print('torch before install:', version('torch'), flush=True)
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
check_code = """
import importlib.metadata, sys, torch, diffusers
print('Fresh process Python:', sys.version, flush=True)
print('Fresh process executable:', sys.executable, flush=True)
for name in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):
    try:
        value = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        value = None
    print(name + ':', value, flush=True)
assert str(torch.__version__) == '2.11.0+cu128', torch.__version__
assert diffusers.__version__ == '0.40.0', diffusers.__version__
"""
subprocess.run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import shutil, subprocess
from pathlib import Path

SOURCE_SHA = '7ad4d9425bfb246b1cd5e4f95aab0c08de521df2'
REPOSITORY = 'https://github.com/RICHAAARC/SC-SSTW.git'
REPO = Path('/content/SC-SSTW-Core-Integration')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_SHA
print('source commit:', actual)

In [ ]:
import subprocess, sys
check_code = """
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('This fixed real Wan run requires a CUDA runtime')
print('device:', torch.cuda.get_device_name(0), flush=True)
"""
subprocess.run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import datetime, os, subprocess, sys
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Video-WM/SC-SSTW-Core-Integration')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE_ROOT / ('integrated_payload_v1_' + stamp)
CONFIG = REPO / 'experiments/wan_state_clock/configs/integrated_payload_v1.json'
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.integrated_payload_run', '--config', str(CONFIG), '--output', str(OUTPUT)]
print('output:', OUTPUT)
subprocess.run(command, cwd=REPO, env=env, check=True)

In [ ]:
import json
result = json.loads((OUTPUT / 'result.json').read_text())
print('status:', result['status'])
print('fixed denominator:', result['fixed_denominator'])
print('calibration:', result['calibration'])
for case_id, case in result['cases'].items():
    print('\n', case_id, case['status'])
    for arm, item in case['videos'].items():
        print(arm, item.get('decision'), item.get('reporting_only'))
print('full result:', OUTPUT / 'result.json')